## Configuration-Driven Satellite Loading

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd()

while not (project_root / "src").is_dir():
    if project_root.parent == project_root:
        raise FileNotFoundError("Could not find project root")

    project_root = project_root.parent

sys.path.insert(0, str(project_root))

In [2]:
import geopandas as gpd
import geemap
import ee

ee.Authenticate()
ee.Initialize()

print("Earth Engine initialized successfully.")

Earth Engine initialized successfully.


In [3]:
from src.areas import load_zones, validate_zones

from src.config import load_config, validate_config

from src.satellite import (
    load_landsat_from_config,

    get_collection_size,
    get_collection_date_range,
    get_first_image,

    get_available_sensors,
    get_sensor_type,
    get_band_mapping,

    validate_collection,
)

from src.preprocessing import (
    mask_clouds,
    mask_snow,
    apply_reflectance_scaling,
    preprocess_collection
)

In [4]:
# Project root
PROJECT_ROOT = Path.cwd().parent

# Configuration file
config_path = PROJECT_ROOT / "config" / "settings.yaml"

# Load and validate configuration
config = load_config(config_path)

validate_config(config)

print("Configuration loaded successfully!")

Configuration loaded successfully!


In [5]:
zones = load_zones()

validate_zones(zones)

print(f"Number of zones: {len(zones)}")
print(zones[["zone_id", "zone_type", "name"]])

study_area = zones.union_all()

study_area_geojson = study_area.__geo_interface__

study_geometry = ee.Geometry(study_area_geojson)

Number of zones: 4
             zone_id     zone_type                                 name
0  aoi_leh_immediate           aoi  Leh town and immediate surroundings
1           urban_01         urban                 Leh urban settlement
2     agriculture_01  agricultural          Irrigated agricultural land
3         natural_01       natural                     Natural mountain


In [6]:
landsat_study = load_landsat_from_config(
    study_geometry=study_geometry,
    config=config,
)

print(
    "Collection valid:",
    validate_collection(landsat_study)
)

print(
    "Number of images:",
    get_collection_size(landsat_study)
)

Collection valid: True
Number of images: 647


## Satellite Metadata Inspection

In [7]:
first_image = get_first_image(
    landsat_study
)

print("Sensor type:", get_sensor_type(first_image))
print("Band mapping:", get_band_mapping(first_image))

print("Date range:", get_collection_date_range(landsat_study))
print("Available sensors:", get_available_sensors(landsat_study))

Sensor type: landsat_457
Band mapping: {'blue': 'SR_B1', 'green': 'SR_B2', 'red': 'SR_B3', 'nir': 'SR_B4', 'swir1': 'SR_B5', 'swir2': 'SR_B7'}
Date range: {'start_date': '1989-08-06', 'end_date': '2025-12-23'}
Available sensors: ['LANDSAT_5', 'LANDSAT_7', 'LANDSAT_8', 'LANDSAT_9']


## Preprocessing pipeline

In [15]:
# Apply the complete preprocessing pipeline.
landsat_preprocessed = preprocess_collection(
    landsat_study,
    "landsat"
)

print("Preprocessing pipeline executed.")
print(landsat_preprocessed)

Preprocessing pipeline executed.
ee.ImageCollection({
  "functionInvocationValue": {
    "functionName": "Collection.map",
    "arguments": {
      "baseAlgorithm": {
        "functionDefinitionValue": {
          "argumentNames": [
            "_MAPPING_VAR_0_0"
          ],
          "body": {
            "functionInvocationValue": {
              "functionName": "Image.addBands",
              "arguments": {
                "dstImg": {
                  "argumentReference": "_MAPPING_VAR_0_0"
                },
                "overwrite": {
                  "constantValue": true
                },
                "srcImg": {
                  "functionInvocationValue": {
                    "functionName": "Image.add",
                    "arguments": {
                      "image1": {
                        "functionInvocationValue": {
                          "functionName": "Image.multiply",
                          "arguments": {
                            "image1": {
   

In [16]:
# Compare collection sizes.

original_count = landsat_study.size().getInfo()

preprocessed_count = (
    landsat_preprocessed.size().getInfo()
)

print("Original image count:", original_count)
print(
    "Preprocessed image count:",
    preprocessed_count
)

Original image count: 647
Preprocessed image count: 647


In [17]:
# Get the first preprocessed image.
preprocessed_first_image = (
    landsat_preprocessed.first()
)

print(
    "Preprocessed bands:",
    preprocessed_first_image.bandNames().getInfo()
)

Preprocessed bands: ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7', 'SR_ATMOS_OPACITY', 'SR_CLOUD_QA', 'ST_B6', 'ST_ATRAN', 'ST_CDIST', 'ST_DRAD', 'ST_EMIS', 'ST_EMSD', 'ST_QA', 'ST_TRAD', 'ST_URAD', 'QA_PIXEL', 'QA_RADSAT']


In [18]:
# Inspect reflectance values from the preprocessed collection.

preprocessed_values = (
    preprocessed_first_image
    .select(["SR_B3", "SR_B4"])
    .reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=study_geometry,
        scale=30,
        maxPixels=1e8
    )
)

print("Preprocessed reflectance values:")
print(preprocessed_values.getInfo())

Preprocessed reflectance values:
{'SR_B3': 0.19723397092082742, 'SR_B4': 0.2731880655503688}
